# Intraday Probability Band Engine

## Backtest Research Notebook

This notebook is the main research and backtesting workspace for the probability band engine.

It loads historical intraday data, applies the reusable `src/` modules, and evaluates VWAP / TWAP reference lines, sigma bands, z-score zones, outcome labels, and empirical probability tables.

The project is structured so that:
- reusable engine logic lives in `src/`
- research, diagnostics, and charts live in this notebook
- live monitoring and trading workflow live in `notebooks/live_trading.ipynb`

---

## What this notebook does

1. Loads and inspects historical intraday data
2. Checks data quality and session structure
3. Computes the intraday reference line
4. Builds sigma bands and z-score zones
5. Extracts context variables
6. Labels historical outcomes
7. Calibrates empirical conditional probabilities
8. Runs backtest analysis and diagnostics
9. Produces charts, tables, and research outputs

---
### Section 0 — Imports and Global Configuration

All dependencies are imported once here. The `CONFIG` dictionary is the **single place
you change parameters** — instrument, file paths, band thresholds, outcome horizon.
Nothing below this cell contains magic numbers.

In [ ]:
from src.config import CONFIG
from src.loaders import load_mt5_csv, load_mt5_live, load_tradingview_csv, load_generic_csv, assign_sessions
from src.reference import compute_reference
from src.sigma import compute_sigma, compute_bands
from src.zones import ZONE_LABELS, ZONE_COLORS, compute_zscore, classify_zone, classify_zones_series
from src.context import compute_context
from src.labelling import label_outcomes
from src.splits import split_by_date
from src.calibration import wilson_ci, calibrate_probability_table
from src.lookup import lookup_probabilities

In [ ]:
print("✅ Configuration loaded")
print(f"   Instrument : {CONFIG['instrument']}")
print(f"   Reference  : {CONFIG['reference_type']}")
print(f"   Vol method : {CONFIG['vol_method']}")
print(f"   Horizon    : {CONFIG['outcome_horizon_bars']} bars")
print(f"   Spread     : {CONFIG['spread_cost']} price units")
print(f"   Edge gap   : {CONFIG['edge_gap_threshold']:.0%} minimum")

---
### Section 1 — Data Normalisation Layer

This is the **only part of the project that knows anything about a specific data source**.
Every loader function below produces an identical output DataFrame with columns:

| Column | Type | Description |
|--------|------|-------------|
| `timestamp` | DatetimeTZ (UTC) | Bar open time |
| `open` | float | Open price |
| `high` | float | High price |
| `low` | float | Low price |
| `close` | float | Close price |
| `volume` | float | Tick volume (proxy for real volume) |
| `typical_price` | float | (H+L+C)/3 — pre-computed for VWAP |
| `session_date` | date | The trading date (for VWAP daily reset) |

Once data passes through a loader, **all downstream code is source-agnostic**.
To add a new source (e.g. Binance, IB, dxFeed), write one new loader function
that outputs this same schema — nothing else changes.# Section 1 — Data Loading and Basic Inspection

In [ ]:
# ── Load data — change loader here if your source changes ──
df = load_mt5_csv(CONFIG['csv_path'])

# ── Basic inspection ──
print(f"\nDate range  : {df['datetime'].iloc[0]} → {df['datetime'].iloc[-1]}")
print(f"Total bars  : {len(df):,}")
print(f"Unique days : {df['session_date'].nunique()}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nFirst 3 rows:")
df.head(3)

### Section 2 — Data Quality Checks

In [ ]:
# ── Data quality checks ──
issues = []

zero_vol = (df['tick_volume'] <= 1.0).sum()
if zero_vol > len(df) * 0.05:
    issues.append(f"⚠️  {zero_vol:,} bars ({zero_vol/len(df):.1%}) have volume ≤ 1 — VWAP may be unreliable")

null_count = df.isnull().sum().sum()
if null_count > 0:
    issues.append(f"⚠️  {null_count} null values found — check loader output")

# Detect time gaps larger than expected bar duration
time_diffs = df['datetime'].diff().dropna()
median_diff = time_diffs.median()
large_gaps = (time_diffs > median_diff * 3).sum()
if large_gaps > 0:
    issues.append(f"⚠️  {large_gaps} time gaps > 3x bar duration (weekends/news expected)")

if issues:
    for i in issues:
        print(i)
else:
    print("✅ Data quality checks passed")

print(f"\nMedian bar duration: {median_diff}")
print(f"Volume stats:\n{df['tick_volume'].describe().round(2)}")

---
### Section 3 — VWAP / Mean Reference Calculation

VWAP is computed **per session** — it resets to zero at the start of each trading day.
This is non-negotiable for intraday work. A multi-day VWAP is a different concept entirely.

**Mathematical definition:**

$$\text{VWAP}_t = \frac{\sum_{i=\text{open}}^{t} P_i^{\text{typical}} \cdot V_i}{\sum_{i=\text{open}}^{t} V_i}$$

Where $P_i^{\text{typical}} = \frac{H_i + L_i + C_i}{3}$

When volume is unavailable or unreliable, **TWAP** (time-weighted) is used as fallback:
$$\text{TWAP}_t = \frac{1}{t} \sum_{i=\text{open}}^{t} P_i^{\text{typical}}$$

The `compute_reference` function automatically selects the correct method based on `CONFIG['reference_type']`.

In [ ]:
df['reference'] = compute_reference(df, CONFIG)
df['price_deviation'] = df['close'] - df['reference']

print(f"\nReference stats (sample):")
print(df[['datetime','close','reference','price_deviation']].tail(5))

---
### Section 4 — Volatility and Sigma Bands

Sigma bands define the "expected" range of price around the reference line.
We use **EWMA (Exponentially Weighted Moving Average) standard deviation** as the primary method
because it updates incrementally without a hard memory cutoff — critical for live mode.

**EWMA variance update equation:**
$$\sigma_t^2 = (1 - \lambda) \cdot r_t^2 + \lambda \cdot \sigma_{t-1}^2$$

Where $r_t = C_t - C_{t-1}$ (price return) and $\lambda = e^{-1/\tau}$ with $\tau$ = half-life in bars.

Band levels are placed at $\text{reference} \pm k\sigma$ for $k \in \{1, 2, 3\}$.

In [ ]:
df['sigma'] = compute_sigma(df, CONFIG)
bands_df = compute_bands(df, df['sigma'])
df = pd.concat([df, bands_df], axis=1)

print("✅ Sigma bands added")
df[['datetime', 'close', 'reference', 'sigma', 'band_1p', 'band_1n']].tail()

---
### Section 5 — Z-Score and Zone Classification

The z-score converts price into a **dimensionless, instrument-agnostic** metric:

$$z_t = \frac{C_t - \text{reference}_t}{\sigma_t}$$

A z-score of +1.8 means the same thing on EURUSD and NAS100 — price is 1.8 sigma above the mean.
This allows the **same zone thresholds and probability tables to be compared across instruments**.

Zones are discrete labels assigned based on configurable z-score thresholds.
Default thresholds: 0.5, 1.0, 2.0 — giving 7 zones (3 each side + mean zone).

In [ ]:
df['z_score'] = compute_zscore(df)
df['zone'] = classify_zones_series(df['z_score'], CONFIG['zone_thresholds'])

print("✅ Z-scores and zones computed")
print(f"\nZone distribution:")
zone_counts = df['zone'].value_counts().reindex(ZONE_LABELS, fill_value=0)
for zone, count in zone_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"  {zone:4s} | {bar:<25} {count:6,} ({pct:.1f}%)")

---
### Section 6 — Context Variables

Raw zone probabilities assume price behaviour is stationary across all conditions.
It isn't. A z-score of +2.0 during a strong uptrend has a very different
mean-reversion probability than the same z-score in a flat, low-volume session.

We compute four context variables and discretise each into bins:
- **Trend**: direction of VWAP slope over the last N bars
- **Volume regime**: current bar volume vs recent average
- **Time of day**: session segment (open, morning, overlap, afternoon, dead)
- **Z-score velocity**: is price accelerating toward or away from the mean?

In [ ]:
ctx_df = compute_context(df, CONFIG)
df = pd.concat([df, ctx_df], axis=1)

---
### Section 7 — Outcome Labelling (Backtest Only)

⚠️ **This section uses future data and must NEVER run in replay or live mode.**

For each bar $t$ we look $N$ bars forward and classify the outcome.
Labels are **cost-adjusted** — an MR label only fires if the reversion
would have produced a net positive result after spread and a realistic stop.

| Outcome | Condition |
|---------|----------|
| **Mean Reversion (MR)** | $|z_{t+N}| < \delta_{MR}$, sign flipped, AND no stop hit AND net edge > min threshold |
| **Continuation (CONT)** | $|z_{t+N}| > |z_t| + \delta_{cont}$ |
| **Neutral (NEU)** | Neither of the above |

These labels are only used to **build the probability table** — never for live signals.

In [ ]:
df['outcome'] = label_outcomes(df, CONFIG, mode='backtest')

---
### Section 8 — Probability Table Calibration

We count how many times each outcome occurred in each zone, then convert to probabilities.
Wilson confidence intervals are used instead of naive standard errors
because they are more accurate for proportions near 0 or 1, and for small sample sizes.

The probability table is the **model artefact** — the output of backtest mode,
and the read-only input to replay and live mode.

---
### Section 8a — Train / Test Split

Historical data is partitioned **strictly by date** into three non-overlapping sets.
Random splits are not used because they leak future session behaviour into training.

| Set | Fraction | Purpose |
|-----|----------|---------|
| Calibration | 60% | Build the probability table |
| Validation | 20% | Tune edge threshold, filters |
| Test | 20% | Final reported numbers — touch once only |

The probability table passed to replay and live mode is always calibrated
on the calibration set only.

In [ ]:
df_cal, df_val, df_tst = split_by_date(df, CONFIG)
print(f"\n⚠️  df_tst must not be used for any tuning decisions.")
print(f"   Use df_cal for calibration, df_val for threshold tuning.")

In [ ]:
# Calibrate on calibration split only
prob_table       = calibrate_probability_table(df_cal, CONFIG)
prob_table_trend = calibrate_probability_table(df_cal, CONFIG, context_col='trend_bin')

print("\nMarginal probability table (MR only):")
mr_only = prob_table[prob_table['outcome'] == 'MR'][
    ['zone','prob','ci_lower','ci_upper','total','confidence']]
print(mr_only.sort_values('zone').to_string(index=False))

---
### Section 9 — Probability Lookup (Used in All Three Modes)

This function is the **runtime interface** to the probability table.
Given a zone and optional context, it returns the three outcome probabilities.
It is called identically in backtest, replay, and live mode.

In [ ]:
# Quick test
test_zone = 'Z2+'
test_ctx  = {'trend_bin': 'flat'}
probs = lookup_probabilities(test_zone, prob_table_trend, prob_table, test_ctx, CONFIG)
print(f"Zone {test_zone} | trend={test_ctx['trend_bin']}")
for o in ['MR', 'CONT', 'NEU']:
    v = probs[o]
    print(f"  {o:4s}: {v['prob']:.1%}  CI [{v['ci_lower']:.1%}–{v['ci_upper']:.1%}]  {v['confidence']}")
print(f"  Edge gap : {probs['edge_gap']:.1%}  |  Dominant: {probs['dominant']}  |  Actionable: {probs['actionable']}")
print(f"  Lookup tier: {probs['lookup_tier']} (1=conditioned, 2=marginal, 3=uniform)")

In [ ]:
for name in ['results', 'state']:
    if name in globals():
        del globals()[name]
print("old objects cleared")